![title](https://i.imgur.com/XPBIVRh.png)
 <!https://imgur.com/a/weIMgzk>
---

<!-- https://imgur.com/a/tItnN59 -->

# Task 8 - Large Language Models (LLMs)

In this exercise, you will explore how to load, configure, and run open-source Large Language Models (LLMs) using **PyTorch** and the **Hugging Face `transformers`** library.

### Objectives:
By the end of this notebook, you will understand how to:
1. Set up your PyTorch environment and inspect hardware acceleration (CUDA/MPS/CPU).
2. Perform **Autoregressive Text Completion** using base autoregressive models (`GPT-2`) and explore decoding strategies (Greedy Search, Top-k, Top-p / Nucleus Sampling, and Temperature).
3. Build an **Interactive Multi-turn Chatbot.** Implement conversation state management (`system`, `user`, `assistant` roles) using an instruction-tuned model (`Qwen2.5-1.5B-Instruct` or `Llama-3.2-1B-Instruct`), apply proper Chat Templates, and build an interactive CLI chat loop
4. Perform **Zero-Shot & Few-Shot Text Classification** (Sentiment Analysis) by leveraging In-Context Learning (ICL) using generative prompting without fine-tuning (updating model weights).
5. Perform **Structured Information Extraction & Summarization** by controlling output structure and generation length. Constrain LLM generation to parse unstructured text into clean, machine-readable JSON dictionaries.
6. Inspect **Raw PyTorch Logits & Probabilities.** Run a direct PyTorch forward pass (`model(**inputs)`) to examine next-token logit distributions and apply `torch.softmax`.
7. Perform **Fine-Tuning Efficiently with LoRA (`peft`).** Freeze base model weights and train lightweight Low-Rank Adaptation matrices on a custom domain dataset.
8. Build **Retrieval-Augmented Generation (RAG) from Scratch.** Compute dense embeddings, perform cosine similarity search via PyTorch matrix multiplication, and ground LLM responses in retrieved documents.

> ## 🗒️ Catatan Penulis — Ringkasan Notebook
> Notebook ini adalah **tur end-to-end tentang cara memakai LLM dengan PyTorch + Hugging Face `transformers`**, dari yang paling dasar (menghasilkan teks) sampai teknik yang dipakai di aplikasi nyata (fine-tuning LoRA dan RAG).
>
> | Part | Topik | Model | Ide utama |
> |---|---|---|---|
> | 0 | Environment | – | Deteksi `cuda` / `mps` / `cpu` dan pilih `dtype` |
> | 1 | Text completion | GPT-2 (124M) | Loop autoregresif; greedy vs sampling |
> | 2 | Chatbot multi-turn | Qwen2.5-1.5B-Instruct | LLM *stateless* → riwayat chat disimpan manual + *chat template* |
> | 3 | Klasifikasi sentimen | Qwen2.5 | Few-shot *In-Context Learning*, tanpa update bobot |
> | 4 | Text → JSON | Qwen2.5 | Menjaga format output agar bisa di-parse program |
> | 5 | Logit & probabilitas | GPT-2 | `model(**inputs)` → `softmax` → `topk` |
> | 6 | LoRA | GPT-2 | Latih <1% parameter, bobot asli dibekukan |
> | 7 | RAG | MiniLM + Qwen2.5 | Embedding + cosine similarity + jawaban yang "membumi" |
>
> Di setiap Part saya tambahkan **📝 Penjelasan Kode** (apa yang dilakukan kode yang baru dilengkapi dan kenapa) dan **📊 Analisis Hasil** (apa arti output yang muncul). Kesimpulan ada di bagian paling akhir.

---
---
# [Part 0] Environment Setup

Before loading any Large Language Model, it is critical to inspect your computing environment. LLMs contain hundreds of millions to billions of parameters. Running them on a CPU is possible for smaller models (like `gpt2` with 124M parameters), whereas a GPU (NVIDIA CUDA or Apple Silicon MPS) dramatically accelerates matrix multiplications.

In addition, modern LLMs often use half-precision floating-point formats such as `torch.float16` or `torch.bfloat16` to cut memory usage in half while maintaining inference accuracy.

1. If an NVIDIA GPU (`torch.cuda.is_available()`) is present, it selects `"cuda"` and `torch.float16`.
2. Else if Apple Silicon MPS (`torch.backends.mps.is_available()`) is present, it selects `"mps"` and `torch.float16`.
3. Otherwise, it falls back to `"cpu"` and `torch.float32` (since many CPUs lack native fast `float16` support).


In [1]:
# Install required libraries if not already installed
!pip install -q torch transformers accelerate peft sentence-transformers

In [2]:
import torch
import transformers

print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")

def get_device_and_dtype():
    if torch.cuda.is_available():
        device = "cuda"
        dtype = torch.float16
    elif torch.backends.mps.is_available():
        device = "mps"
        dtype = torch.float16
    else:
        device = "cpu"
        dtype = torch.float32
    return device, dtype

device, dtype = get_device_and_dtype()
print(f"Using device: {device} | Using dtype: {dtype}")

PyTorch Version: 2.14.1+cu130
Transformers Version: 5.22.1
Using device: cuda | Using dtype: torch.float16


### Analisis Hasil — Part 0
- Runtime memakai **GPU NVIDIA (`cuda`)** dengan **`float16`**, sehingga model 1,5 miliar parameter (Qwen) muat dan berjalan cepat. Pada CPU, `dtype` otomatis jatuh ke `float32`.
- Versi yang dipakai: PyTorch `2.14.1+cu130` dan Transformers `5.22.1`. Ini penting karena Transformers 5.x menandai beberapa argumen lama sebagai *deprecated* (contoh: `torch_dtype` → `dtype`, lihat Part 2).
- Peringatan *"unauthenticated requests to the HF Hub"* **tidak berbahaya**; hanya berarti kita belum login Hugging Face sehingga batas unduhan lebih rendah.

If you already have a HuggingFace account, uncomment these code and run it to login and enable higher rate limits and faster downloads.

In [3]:
# from huggingface_hub import login

# login()

---
---
# [Part 1] Autoregressive Text Completion with GPT-2


**GPT-2** (Generative Pre-trained Transformer 2) is a classic causal (autoregressive) language model trained to predict the next token given all previous tokens in a sequence. Unlike modern instruction-tuned models, base GPT-2 simply continues whatever text prompt it is given.




---
## 1 - Initialize Model

Because `gpt2` (124M parameters) is lightweight, ungated, and fast to download, it is ideal for understanding the fundamentals of LLM inference in PyTorch.

To run inference, we need two components from Hugging Face `transformers`:

* **Tokenizer (`AutoTokenizer`)**: Converts raw text strings into numerical token IDs that the model can process, and decodes output IDs back into readable text.
* **Model (`AutoModelForCausalLM`)**: Loads the pre-trained neural network weights with a causal language modeling head for next-token prediction.

---
#### <font color='red'>**EXERCISE:** </font>

    Load the tokenizer and causal LM model for GPT-2, then move model to `device`


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID_GPT2 = "gpt2"

# call AutoTokenizer.from_pretrained() with input model ID
tokenizer_gpt2 = AutoTokenizer.from_pretrained(MODEL_ID_GPT2)

# call AutoModelForCausalLM.from_pretrained() with input ID name, then call .to(device)
model_gpt2 = AutoModelForCausalLM.from_pretrained(MODEL_ID_GPT2).to(device)

# switche the model to inference mode
model_gpt2.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### Penjelasan Kode — Memuat GPT-2
- `AutoTokenizer.from_pretrained("gpt2")` mengunduh kamus subword (BPE, **50.257 token**) dan aturan pemecahan teks.
- `AutoModelForCausalLM.from_pretrained("gpt2")` mengunduh arsitektur (12 blok Transformer, dimensi 768) beserta bobot terlatih; `.to(device)` memindahkannya ke GPU.
- `model.eval()` mematikan perilaku khusus training (terutama **Dropout**), sehingga hasil inferensi konsisten. Output sel berupa cetakan arsitektur model, bukan error.

---
## 2 - Inference Function



When generating text, the model runs an **autoregressive loop** made up of four stages:

1. **Tokenization (`tokenizer`):** Raw text is split into subword tokens and mapped to integer IDs (`input_ids`) alongside an `attention_mask` packed into PyTorch tensors of shape `(batch_size, sequence_length)`<br> (a tensor of `1`s and `0`s indicating which tokens are real vs. padding). These tensors must be moved to `device`.
2. **Forward Pass:** On each step of the loop, the model processes the current sequence and outputs raw prediction scores (**logits**) over the entire vocabulary (50,257 tokens for GPT-2), which are converted into probabilities for the next token.
3. **Decoding Strategy:** Selects the next token from the probabilities, appends it to the sequence, and repeats until `max_new_tokens` (or an end-of-sequence token) is reached:
   * **Greedy Search (`do_sample=False`):** <br>Always picks the single highest-probability token (`argmax`). Deterministic and accurate for facts, but prone to repetitive loops in open-ended writing.
   * **Sampling (`do_sample=True`):** <br>Randomly samples the next token according to the probability distribution. When `do_sample=True`, you can shape the distribution using:
     * **Temperature (`temperature`):** <br>Scales raw logits before softmax. <br>Lower values (`< 1.0`) make the distribution sharper and more focused <br>higher values (`> 1.0`) increase creativity and randomness.
     * **Top-k Sampling (`top_k`):** <br>Restricts sampling to the $k$ most likely next tokens, cutting off the long tail of unlikely words.
     * **Nucleus / Top-p Sampling (`top_p`):** <br>Dynamically selects the smallest set of top tokens whose cumulative probability exceeds $p$ (e.g., `0.92`).
4. **Decoding to Text (`tokenizer.decode`):** <br>Converts the generated token IDs back into a human-readable string (typically with `skip_special_tokens=True`).


---
#### <font color='red'>**EXERCISE:** </font>
<pre>
Complete the inference function below by tokenizing the input prompt,
moving the tensors to `device`, generating tokens with `model_gpt2.generate()`,
and decoding the output back into text.


In [5]:
def generate_text(model, tokenizer, prompt,
                  max_new_tokens=50, do_sample=True,
                  temperature=0.8, top_k=50, top_p=0.92,
                  with_input=True):

    # Tokenize and move to device
    # Pass prompt and return_tensor="pt" through tokenizer(),
    # then call .to(device)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Define parameters
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample"     : do_sample,
        "pad_token_id"  : tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_k"]       = top_k
        gen_kwargs["top_p"]       = top_p

    # Generate continuation tokens without tracking gradients
    with torch.no_grad():
        # Pass **inputs and **gen_kwargs through model.generate()
        outputs = model.generate(**inputs, **gen_kwargs)

    # return with or without initial input prompt
    if with_input:
        input_length = 0
    else:
        # get tokenized input prompt length
        input_length = inputs["input_ids"].shape[1]

    # Slice only newly generated tokens
    new_tokens = outputs[0][input_length:]

    # Decode the generated token IDs back into a human-readable string
    # Pass new_tokens through self.tokenizer.decode() using skip_special_tokens=True
    # then call .strip()
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return generated_text


### Penjelasan Kode — Fungsi `generate_text`
Fungsi ini membungkus seluruh pipeline inferensi:
1. **Tokenisasi**: `tokenizer(prompt, return_tensors="pt").to(device)` → tensor `input_ids` dan `attention_mask` berbentuk `(1, panjang_prompt)` di GPU.
2. **Generasi**: `model.generate(**inputs, **gen_kwargs)` menjalankan loop autoregresif (prediksi token → tempel ke urutan → ulangi). Dibungkus `torch.no_grad()` agar tidak menyimpan gradien (hemat memori).
3. **Memotong prompt**: hasil `generate` berisi *prompt + token baru*. Bila `with_input=False`, kita iris `outputs[0][input_length:]` agar hanya jawaban baru yang tersisa. Ini **wajib** untuk chatbot, klasifikasi, dan JSON, supaya yang kita dapat hanya jawaban model.
4. **Decode**: `tokenizer.decode(..., skip_special_tokens=True).strip()` mengubah ID kembali menjadi teks bersih.

`pad_token_id = eos_token_id` dipasang karena GPT-2 tidak punya token padding bawaan; tanpa ini Transformers akan memberi peringatan. Parameter `temperature/top_k/top_p` hanya diberikan saat `do_sample=True`, sebab pada greedy parameter tersebut tidak dipakai.

---
## 3 - Test Generation

Check your implementation

In [6]:
prompt = "Artificial intelligence in university education will transform"
print("--- Greedy Search (Deterministic) ---")
print(generate_text(model_gpt2, tokenizer_gpt2, prompt, do_sample=False))

print("\n--- Creative Sampling (temp=0.9, top_p=0.9) ---")
print(generate_text(model_gpt2, tokenizer_gpt2, prompt, do_sample=True, temperature=0.9, top_p=0.9))


--- Greedy Search (Deterministic) ---
Artificial intelligence in university education will transform the way we think about the world.

The future of education is not just about the future of science, technology, engineering and math. It is also about the future of the world.

The future of education is not just about the

--- Creative Sampling (temp=0.9, top_p=0.9) ---
Artificial intelligence in university education will transform the university into a world-class research and creative place.

"I think it's a great opportunity to work with universities across the globe, which is what we're doing at the University of Edinburgh. We're not just working on the AI


**Expected Result**:
<pre>
Ouytput should be something like:

--- Greedy Search (Deterministic) ---
Artificial intelligence in university education will transform the way we think about the world....

--- Creative Sampling (temp=0.9, top_p=0.9) ---
Artificial intelligence in university education will transform the way universities teach, provide for, and employ students....

### 📊 Analisis Hasil — Part 1
**Greedy search** menghasilkan teks yang *koheren di awal tetapi berulang*: kalimat "The future of education is not just about…" muncul berkali-kali dan teks terpotong di tengah kalimat karena batas `max_new_tokens=50`. Ini adalah **kelemahan klasik greedy decoding**: karena selalu memilih token ber-probabilitas tertinggi, model mudah terjebak dalam loop. Hasilnya deterministik (dijalankan ulang akan sama persis, dan memang sama dengan run sebelumnya).

**Sampling** (`temperature=0.9, top_p=0.9`) menghasilkan teks yang *lebih bervariasi dan tidak berulang*, tetapi **tidak stabil**. Pada run ini awalnya relevan ("transform the university into a world-class research and creative place"), lalu berlanjut ke kutipan karangan tentang kerja sama dengan universitas dunia dan "University of Edinburgh" yang tidak ada dasarnya. Pada run sebelumnya hasilnya bahkan membahas "mental illness and dementia". Isi sampling **berbeda di setiap run**, dan ini normal.

**Pelajaran:** tidak ada strategi decoding yang "paling benar". *Greedy* cocok untuk tugas faktual/terstruktur (klasifikasi, JSON, QA), sedangkan *sampling* cocok untuk tulisan kreatif. Selain itu GPT-2 adalah model kecil yang hanya meneruskan teks, sehingga mengarang kutipan dan fakta itu wajar. Contoh "Expected Result" hanya ilustrasi.

---
---
# [Part 2] Multi-Turn Chatbot with Instruction-Tuned LLMs

Base models like GPT-2 are trained purely to predict and continue text. <br> i.e., If you prompt a base model with *"What is the capital of France?"*, it might simply continue with *"What is the capital of Germany?"* <br>because it assumes it is completing a list of trivia questions.

To build a helpful **Chatbot**, we use an **Instruction-Tuned (Chat) Model** that has undergone Supervised Fine-Tuning (SFT) and alignment techniques like Reinforcement Learning from Human Feedback (RLHF) or Direct Preference Optimization (DPO) to follow user instructions.

Popular lightweight examples include **Meta Llama-3.2-1B-Instruct** and **Qwen2.5-1.5B-Instruct**.


---
## 1 - Initialize Chat Model

Meta's Llama-3 models are gated and require accepting a license agreement on Hugging Face along with an access token. In this notebook, we default to `Qwen/Qwen2.5-1.5B-Instruct` so the code runs immediately without authentication barriers, while using the exact same Hugging Face `transformers` API as `meta-llama/Llama-3.2-1B-Instruct`.

> **Tip:** Instruction-tuned models still use `AutoTokenizer` and `AutoModelForCausalLM` just like base models, the key difference lies in how they were trained and how we format prompts using **Chat Templates** (covered next).

---
#### <font color='red'>**EXERCISE:** </font>

    Load the tokenizer and causal LM model for Qwen chat model, then move model to `device`


In [7]:
# Note: You can swap this with "meta-llama/Llama-3.2-1B-Instruct" if you have HF access token configured
CHAT_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# call AutoTokenizer.from_pretrained() with input model ID
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID)

# call AutoModelForCausalLM.from_pretrained() with input ID name and dtype=dtype
# then call .to(device)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL_ID, dtype=dtype).to(device)

chat_model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

### Penjelasan Kode — Memuat Model Chat
Mekanismenya sama dengan GPT-2; bedanya model ini sudah di-*instruction-tune* (SFT + alignment) sehingga **mengikuti perintah** alih-alih sekadar melanjutkan teks. Argumen `dtype=dtype` memuat bobot dalam `float16` di GPU (≈3 GB, separuh dari `float32`). *Catatan:* pada kode awal tertulis `torch_dtype=`; di Transformers 5.x argumen ini *deprecated* sehingga saya ganti ke `dtype=` (perilakunya identik).

---
## 2 - Multi-Turn Conversation Memory
LLMs are **stateless** between forward passes. they have no built-in memory of past messages.

To build a chatbot, our Python application must maintain a chronological list of message dictionaries:
```python
messages = [
    {"role": "system", "content": "You are a helpful PyTorch teaching assistant."},
    {"role": "user", "content": "What is a tensor?"},
    {"role": "assistant", "content": "A tensor is a multi-dimensional array..."},
    {"role": "user", "content": "How do I move it to a GPU?"}
]
```



To manage this cleanly across turns, we wrap the logic inside a `Chatbot` class that initializes `self.history` with the `system` prompt and updates the log on every turn:

1. **Append User Message:** <br>Add `{"role": "user", "content": user_input}` to `self.history`.
2. **Apply Chat Template:** <br>Call `tokenizer.apply_chat_template(self.history, tokenize=False, add_generation_prompt=True)` to format the history with model-specific control tokens  and signal that it is the assistant's turn to reply. <br>(such as `<|im_start|>` for Qwen or `<|start_header_id|>` for Llama)
3. **Tokenize & Generate:** <br>Tokenize the formatted prompt string and generate the reply using out previous `generate_text()` function.
4. **Slice & Decode New Tokens:** <br>Slice the generated tensor and decode it with `with_input=False` so we only extract the **newly generated** assistant reply, rather than repeating the entire conversation history.
5. **Append Assistant Reply:** <br>Add `{"role": "assistant", "content": reply}` back to `self.history` so subsequent turns retain full context.


---
#### <font color='red'>**EXERCISE:** </font>

    Complete the `Chatbot` class below
    by initializing the conversation history with the system prompt
    and implementing the turn-by-turn response method.



In [8]:
class PyTorchChatbot:
    def __init__(self, model, tokenizer, system_prompt="You are a helpful, concise AI tutor."):
        self.model     = model
        self.tokenizer = tokenizer
        self.history   = [{"role": "system", "content": system_prompt}]

    def chat(self, user_message, max_new_tokens=150, temperature=0.7):
        # Add user message to history
        self.history.append({"role": "user", "content": user_message})

        # Apply chat template
        # Pass self.history, tokenize=False, and add_generation_prompt=True
        # through self.tokenizer.apply_chat_template()
        formatted_prompt = self.tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        )

        # Call generate_text() with input self.model, self.tokenizer, and formatted_prompt
        # Pass max_new_tokens, temperature from input function
        # set do_sample=True, top_p=0.9, and with_input=False
        reply = generate_text(self.model, self.tokenizer, formatted_prompt,
                              max_new_tokens=max_new_tokens,
                              do_sample=True,
                              temperature=temperature,
                              top_p=0.9,
                              with_input=False)

        # Add assistant response to history
        self.history.append({"role": "assistant", "content": reply})

        return reply

    # Dipakai oleh perintah "/reset" pada run_interactive_chat()
    def reset(self):
        self.history = self.history[:1]  # sisakan system prompt saja


### Penjelasan Kode — Kelas `PyTorchChatbot`
LLM itu **stateless**: tiap pemanggilan tidak mengingat apa pun. "Ingatan" chatbot sebenarnya hanyalah **list `self.history`** yang kita kirim ulang secara utuh setiap giliran.

Alur `chat()`:
1. Tambahkan pesan `user` ke `history`.
2. `apply_chat_template(..., tokenize=False, add_generation_prompt=True)` mengubah list pesan menjadi satu string dengan token kontrol milik Qwen (`<|im_start|>system … <|im_end|>`) dan menutupnya dengan penanda "giliran assistant".
3. Panggil `generate_text(..., with_input=False)` untuk mendapat **hanya** balasan baru.
4. Simpan balasan sebagai pesan `assistant` agar giliran berikutnya punya konteks penuh.

Saya juga menambahkan method **`reset()`** (`self.history[:1]`, hanya menyisakan *system prompt*). Method ini **tidak ada di kode soal**, padahal `run_interactive_chat` memanggil `chatbot.reset()` saat user mengetik `/reset`; tanpa method ini perintah `/reset` akan melempar `AttributeError`.

---
## 3 - Test Chatbot
Check your implementation

In [9]:
bot = PyTorchChatbot(chat_model, chat_tokenizer)

print("You: Hi! I am studying Computer Science in Bandung.")
print("Assistant:", bot.chat("Hi! I am studying Computer Science in Bandung."))
print()
print("You: What am I studying and where am I located?")
print("Assistant:", bot.chat("What am I studying and where am I located?"))


You: Hi! I am studying Computer Science in Bandung.
Assistant: Hello! Studying computer science at a university like ITB (Indonesia Institute of Technology Bandung) is a great choice for developing your skills and knowledge in this field. If you have any specific questions or need guidance on courses, resources, or career advice, feel free to ask. Good luck with your studies!

You: What am I studying and where am I located?
Assistant: As an AI language model, I don't know your exact location or what program you're enrolled in. However, since you mentioned "Computer Science" and being in "Bandung," it's likely that you're attending a university in Indonesia.

If you could provide more details about the specific institution and program you're studying, I'd be happy to help you further. Is there anything else you'd like to discuss related to your studies or general academic advice?


**Expected Result**:
<pre>
Output should be something like:

You: Hi! I am studying Computer Science in Bandung.
Assistant: Hello! Studying computer science in Bandung sounds like an exciting journey...

You: What am I studying and where am I located?
Assistant: If you're currently enrolled in a Bachelor of Science program, then you are likely studying software engineering...

### 📊 Analisis Hasil — Part 2
- **Giliran 1:** model menyambut dan menyebut "ITB (Indonesia Institute of Technology Bandung)". Ini **kekeliruan kecil**: ITB sebenarnya *Institut Teknologi Bandung*, dan Anda tidak pernah menyebut ITB. Model "menebak" kampus dari kata Bandung. Contoh halusinasi ringan pada model kecil.
- **Giliran 2:** model menjawab *"As an AI language model, I don't know your exact location or what program you're enrolled in"*, **tetapi** lalu menulis *"since you mentioned 'Computer Science' and being in 'Bandung', it's likely that you're attending a university in Indonesia"*. Jadi **memori percakapan tetap bekerja**: model hanya bisa tahu kata "Computer Science" dan "Bandung" karena riwayat giliran 1 ikut dikirim. Namun jawabannya *ragu-ragu dan kurang langsung* (tidak sekadar menjawab "Anda belajar Computer Science di Bandung"). Hasil ini berbeda dari run sebelumnya, yang menjawab langsung, karena sampling `temperature=0.7` membuat jawaban berubah-ubah di tiap run.
- Konsekuensi desain "ingatan = riwayat": **panjang prompt bertambah di tiap giliran**, sehingga biaya/latensi naik dan suatu saat menabrak batas *context window*. Aplikasi nyata biasanya memangkas atau meringkas riwayat lama.
- Output berbeda dari contoh di soal; itu aman karena contohnya hanya ilustrasi ("something like").

---
## 4 - Interactive Inference Function

To make our chatbot usable as a real terminal/notebook application, we wrap `chatbot.chat(...)` inside a `while True` interaction loop that:
* Continuously reads user prompts via Python's built-in `input("You: ")`.
* Supports special commands such as `/exit` or `quit` to break out of the loop cleanly, and `/reset` to clear conversation memory back to the initial system prompt.
* Gracefully handles empty inputs or `KeyboardInterrupt` (`Ctrl+C`).

In [10]:
def run_interactive_chat(chatbot):
    print("=" * 60)
    print("Interactive PyTorch LLM Chatbot Started!")
    print("Commands: type '/reset' to clear memory, or 'exit' / 'quit' to stop.")
    print("=" * 60)

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nSession interrupted. Goodbye!")
            break

        if not user_input:
            continue

        if user_input.lower() in {"exit", "quit", "/exit"}:
            print("Assistant: Goodbye! Happy coding with PyTorch.")
            break

        if user_input.lower() == "/reset":
            chatbot.reset()
            print("[System: Conversation history cleared.]")
            continue

        response = chatbot.chat(user_input)
        print(f"Assistant: {response}")

---
## 5 - Test Interactive Chatbot

In [11]:
run_interactive_chat(bot)

Interactive PyTorch LLM Chatbot Started!
Commands: type '/reset' to clear memory, or 'exit' / 'quit' to stop.

You: exit
Assistant: Goodbye! Happy coding with PyTorch.


### Catatan — Chat Interaktif
Pada run ini sel hanya diuji dengan langsung mengetik `exit`, jadi yang terbukti baru **loop dan penanganan perintah keluar**. Perintah `/reset` dan percakapan multi-turn interaktif belum teruji. Untuk uji yang lebih meyakinkan: ketik pesan yang berisi fakta (mis. nama Anda), tanya ulang, ketik `/reset`, lalu tanya lagi. Setelah reset model seharusnya **tidak lagi ingat** fakta tersebut. Sel `input()` ini memang memblokir eksekusi sampai user mengetik, jadi tidak cocok untuk *Run All* otomatis.

---
---
# [Part 3] Zero-Shot & Few-Shot Text Classification

Traditionally, text classification (such as sentiment analysis or topic routing) required fine-tuning a dedicated classification head (`AutoModelForSequenceClassification`) on thousands of labeled examples.

With modern LLMs, we can perform **In-Context Learning (ICL)**, guiding the model to classify text purely through prompt design without updating any weights:

* **Zero-Shot Prompting:** <br>We describe the task and valid labels directly in the prompt without providing any examples.
* **Few-Shot Prompting:** <br>We include 2–3 input/output examples inside the prompt so the model strictly follows our desired output format (e.g., outputting only a single label: `POSITIVE`, `NEGATIVE`, or `NEUTRAL`).
* **Deterministic Decoding:** <br>For classification tasks, we set `do_sample=False` (greedy decoding) so the model always picks the highest-probability label, and use a small `max_new_tokens` (e.g., `5` to `10`) to ensure a concise, easily parseable response while speeding up inference.




---
#### <font color='red'>**EXERCISE:** </font>

    Complete the classification function below
    by building a prompt (supporting both zero-shot and few-shot examples)
    and generating a deterministic label with the chat model.


In [12]:
def classify_sentiment(review_text):

    # define few-show example prompts
    messages = [
        {"role": "system",
         "content": "You are a precise sentiment classifier. Classify the user's review into exactly one of these labels: POSITIVE, NEGATIVE, or NEUTRAL. Output only the single label word and nothing else."},
        {"role": "user", "content": "I absolutely love the sound quality of these headphones!"},
        {"role": "assistant", "content": "POSITIVE"},
        {"role": "user", "content": "The software crashes every five minutes and support never replies."},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "The book has 300 pages and was published in 2023."},
        {"role": "assistant", "content": "NEUTRAL"},
        {"role": "user", "content": review_text}
    ]

    # Apply chat template
    # Pass messages, tokenize=False, and add_generation_prompt=True
    # through chat_tokenizer.apply_chat_template()
    formatted_prompt = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Call generate_text() with input chat_model, chat_tokenizer, and formatted_prompt
    # set max_new_tokens=5, do_sample=False, and with_input=False
    label = generate_text(chat_model, chat_tokenizer, formatted_prompt,
                          max_new_tokens=5, do_sample=False, with_input=False)

    return label

### Penjelasan Kode — `classify_sentiment`
Tidak ada pelatihan; kita "mengajari" model lewat **prompt**:
- **System prompt** menetapkan tugas dan label yang valid (`POSITIVE / NEGATIVE / NEUTRAL`) serta perintah *hanya keluarkan satu kata*.
- Tiga pasangan `user → assistant` berfungsi sebagai **few-shot examples** (satu per kelas), sehingga model meniru formatnya (**In-Context Learning**).
- Ulasan baru ditaruh paling akhir sebagai pesan `user`.
- `do_sample=False` (greedy) membuat hasil deterministik, dan `max_new_tokens=5` cukup untuk satu kata label sekaligus mempercepat inferensi.

Let's test your implementation

In [13]:
sample_reviews = [
    "The battery life on this laptop easily lasts 12 hours during lectures!",
    "The package arrived three days late and the screen was cracked.",
    "It is an average wireless mouse. It works as expected, nothing special."
]

for review in sample_reviews:
    print(f"Review: {review}\nPredicted Sentiment: {classify_sentiment(review)}\n")


Review: The battery life on this laptop easily lasts 12 hours during lectures!
Predicted Sentiment: POSITIVE

Review: The package arrived three days late and the screen was cracked.
Predicted Sentiment: NEGATIVE

Review: It is an average wireless mouse. It works as expected, nothing special.
Predicted Sentiment: NEUTRAL



**Expected Result**:
<pre>
Predicted Sentiment: POSITIVE
Predicted Sentiment: NEGATIVE
Predicted Sentiment: NEUTRAL

### Analisis Hasil — Part 3
Ketiga prediksi **benar sesuai Expected Result** (POSITIVE, NEGATIVE, NEUTRAL). Kasus ketiga ("average… nothing special") paling menarik karena tidak punya kata emosional yang kuat; contoh *NEUTRAL* di few-shot membantu model memilih label netral. Keunggulan pendekatan ini: **tanpa dataset berlabel besar dan tanpa training**. Kelemahannya: hanya 3 contoh uji sehingga belum cukup untuk mengklaim akurasi tinggi; untuk evaluasi serius perlu dataset uji yang lebih banyak dan mengukur akurasi/F1. Output juga belum divalidasi: bila model mengeluarkan kata di luar tiga label, kodenya tidak akan menolaknya.

---
---
# [Part 4] Structured Information Extraction (Text-to-JSON)

In real-world software engineering, LLM outputs often need to be parsed by downstream Python code rather than read by humans. One of the most valuable practical skills in applied NLP is **Structured Information Extraction**.

A major enterprise use case for that is prompting an LLM to extract specific entities from unstructured text and return valid JSON.

### Key Engineering Techniques

* **Schema Specification:** <br>Explicitly define the exact JSON keys, expected data types, and fallback behavior (e.g., using `null` if a field is missing) in the system prompt, and instruct the model to return *only* raw JSON without conversational filler.
* **Deterministic Generation:** <br>Use greedy decoding (`do_sample=False`) to minimize syntax errors or hallucinated keys.
* **Post-Processing Cleanup:** <br>Smaller models occasionally wrap JSON output inside Markdown code fences (e.g., ```` ```json ... ``` ````). <br>Stripping these wrappers before calling `json.loads()` ensures our Python parser succeeds reliably.


---
#### <font color='red'>**EXERCISE:** </font>

    Complete the extraction function below
    by instructing the model to output a strict JSON schema,
    cleaning any Markdown wrappers from the response,
    and parsing the result into a Python dictionary with `json.loads()`.


In [14]:
import json

def extract_course_info(text):

    # Create messages instructing the model to return ONLY raw JSON with keys:
    messages = [
        {"role": "system",
         "content": (
            'Extract structured data from the user\'s text and respond ONLY with a valid JSON object. '
            'Do not wrap the JSON in markdown backticks. Use these exact keys: '
            '"course_code" (string), "course_title" (string), "instructor" (string), "credits" (integer).'
        )},
        {"role": "user", "content": text}
    ]

    # Apply chat template
    # Pass messages, tokenize=False, and add_generation_prompt=True
    # through chat_tokenizer.apply_chat_template()
    formatted_prompt = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Call generate_text() with input chat_model, chat_tokenizer, and formatted_prompt
    # set max_new_tokens=120, do_sample=False, and with_input=False
    raw_output = generate_text(chat_model, chat_tokenizer, formatted_prompt,
                              max_new_tokens=120, do_sample=False, with_input=False)

    # Clean potential markdown fences just in case
    raw_output = raw_output.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    return json.loads(raw_output)


### Penjelasan Kode — `extract_course_info`
Ide utamanya: **memaksa output LLM menjadi data yang bisa dipakai program**.
1. System prompt menentukan skema persis (kunci `course_code`, `course_title`, `instructor`, `credits` beserta tipenya) dan melarang pembungkus markdown.
2. Greedy decoding dan `max_new_tokens=120` menjaga output singkat dan stabil.
3. **Post-processing**: model kecil kadang tetap membungkus JSON dengan ```` ```json ````, jadi dibersihkan dengan `removeprefix/removesuffix` sebelum `json.loads()`.
4. `json.loads()` mengubah string menjadi `dict` Python. Bila model menghasilkan JSON rusak, baris ini akan melempar `JSONDecodeError`, sehingga pada produksi perlu `try/except` dan percobaan ulang.

Let's test your implementation

In [15]:
announcement_text = """
Attention all third-year students! Registration is now open for CS-402: Deep Learning for Natural Language Processing.
This 4-credit advanced elective will be taught by Dr. Elena Rostova every Tuesday and Thursday morning.
"""

result = extract_course_info(announcement_text)
print("Extracted Python Dict:", result)
print("Course Code:", result["course_code"])
print("Instructor :", result["instructor"])

Extracted Python Dict: {'course_code': 'CS-402', 'course_title': 'Deep Learning for Natural Language Processing', 'instructor': 'Dr. Elena Rostova', 'credits': 4}
Course Code: CS-402
Instructor : Dr. Elena Rostova


**Expected Result**:
<pre>
Extracted Python Dict: {'course_code': 'CS-402', 'course_title': 'Deep Learning for Natural Language Processing', ...
Course Code: CS-402
Instructor : Dr. Elena Rostova

### Analisis Hasil — Ekstraksi #1 (kasus mudah)
Seluruh 4 field benar dan tipenya sesuai: `course_code='CS-402'`, `course_title='Deep Learning for Natural Language Processing'`, `instructor='Dr. Elena Rostova'`, `credits=4` (bertipe **integer**, bukan string "4-credit"). Artinya model bukan sekadar menyalin teks, tetapi juga **menormalkan tipe data** sesuai skema.

Let's test it with a more challenging prompt

In [16]:
announcement_text = """
Attention students! Next Tuesday, November 18, 2026, our CS-112 Deep Learning guest lecture
on 'Transformer Attention Kernels' will be held in Auditorium Hall B instead of the usual lab.
"""

result = extract_course_info(announcement_text)
print("Extracted Python Dict:", result)
print("Course Code :", result["course_code"])
print("Course Title:", result["course_title"])

Extracted Python Dict: {'course_code': 'CS-112', 'course_title': "'Transformer Attention Kernels'", 'instructor': 'Not specified in the given text.', 'credits': 3}
Course Code : CS-112
Course Title: 'Transformer Attention Kernels'


**Expected Result**:
<pre>
Extracted Python Dict: {'course_code': 'CS-112', 'course_title': "'Transformer Attention Kernels'", 'instructor': 'Not specified...
Course Code : CS-112
Course Title: 'Transformer Attention Kernels'

### Analisis Hasil — Ekstraksi #2 (kasus sulit)
- `course_code='CS-112'` dan `course_title="'Transformer Attention Kernels'"` benar (tanda kutip tunggal ikut terbawa persis seperti di Expected Result).
- `instructor='Not specified in the given text.'` → informasi memang tidak ada, tapi model **mengisi kalimat bebas** pada field yang seharusnya string nama. Kode yang menganggap field itu nama orang akan salah.
- `credits=3` adalah **halusinasi**: pengumuman tidak menyebut SKS sama sekali. Model "menebak" agar skema terisi.

**Penyebab:** system prompt kita tidak menjelaskan apa yang harus dilakukan saat data tidak ada. Sesuai teknik di teks soal (*schema specification with fallback behavior*), perbaikannya adalah menambahkan instruksi seperti:
```python
'If a field is not mentioned in the text, set its value to null. Never guess or invent values.'
```
Alternatif lain: memvalidasi hasil dengan skema (mis. `pydantic`) dan *constrained decoding*. **Pelajaran:** output yang "terlihat valid" belum tentu benar. Validasi isi tetap diperlukan.

---
---
# [Part 5] Inspecting Next-Token Probabilities


So far, we have used the high-level `.generate()` helper method. However, because Hugging Face models are standard `torch.nn.Module` subclasses, you can pass tokenized inputs directly into `model(**inputs)` to inspect the raw output **logits** and extract a probability distribution over the entire vocabulary for the next token!

### Tensor Shapes in Causal Language Models

When we pass `inputs` of shape `(batch_size, seq_len)` into `model_gpt2`:

* `outputs.logits` is a PyTorch tensor of shape `(batch_size, seq_len, vocab_size)`.
* For GPT-2, `vocab_size = 50257`.
* The vector `outputs.logits[0, -1, :]` contains the raw, unnormalized scores predicting the token that comes **immediately after** the final token in our prompt.
* Applying `torch.softmax(next_token_logits, dim=-1)` converts those raw logits into a valid probability distribution summing to `1.0` (which can then be ranked with `torch.topk()` and decoded back to text).



---
#### <font color='red'>**EXERCISE:** </font>

    Complete the function below to run a forward pass without gradient tracking,
    extract the next-token probability distribution using `torch.softmax`,
    and return the top-k most likely next tokens along with their probabilities.



In [17]:
def get_top_next_tokens(prompt, top_k=5):

    # Pass prompt and return_tensor="pt" through tokenizer_gpt2(), then call .to(device)
    inputs = tokenizer_gpt2(prompt, return_tensors="pt").to(device)

    # Run forward pass without tracking gradients
    with torch.no_grad():
        # Pass **inputs through model_gpt2()
        outputs = model_gpt2(**inputs)

    # Get logits for the final token
    next_token_logits = outputs.logits[0, -1, :]

    # convert to probabilities via softmax
    # Pass next_token_logits through torch.softmax() with dim=-1
    probs = torch.softmax(next_token_logits, dim=-1)

    # Extract top_k
    # Pass probs through torch.topk() with k=tok_k
    top_probs, top_indices = torch.topk(probs, k=top_k)

    results = []
    for prob, idx in zip(top_probs, top_indices):
        # Decode each index in top_indices
        # Pass [idx.item()] through tokenizer_gpt2.decode()
        token_str = tokenizer_gpt2.decode([idx.item()])

        # append results
        results.append((token_str, prob.item()))

    return results

### Penjelasan Kode — `get_top_next_tokens`
Ini membuka "kotak hitam" `generate()`:
- `model_gpt2(**inputs)` menjalankan **satu forward pass**. `outputs.logits` berbentuk `(batch, seq_len, vocab_size)` = `(1, n, 50257)`.
- `logits[0, -1, :]` mengambil skor untuk posisi **terakhir**, yaitu prediksi token yang akan muncul *setelah* prompt.
- `torch.softmax(..., dim=-1)` mengubah skor mentah menjadi distribusi probabilitas (jumlah = 1).
- `torch.topk(probs, k)` mengambil k probabilitas terbesar beserta indeksnya, lalu `decode([idx])` mengubah indeks jadi kata.
(Di komentar soal tertulis `k=tok_k` dan `inputs_ids`; itu salah ketik, kode memakai `top_k` dan `input_ids`.)

Let's test your implementation

In [18]:
predictions = get_top_next_tokens("The capital of Indonesia is", top_k=5)
for token, prob in predictions:
    print(f"Token: {token!r:15} | Probability: {prob:.4f}")

Token: ' the'          | Probability: 0.0965
Token: ' a'            | Probability: 0.0541
Token: ' now'          | Probability: 0.0365
Token: ' located'      | Probability: 0.0351
Token: ' also'         | Probability: 0.0320


**Expected Result**:
<pre>
Token: ' the'          | Probability: 0.0965
Token: ' a'            | Probability: 0.0541
Token: ' now'          | Probability: 0.0365
Token: ' located'      | Probability: 0.0351
Token: ' also'         | Probability: 0.0320

### Analisis Hasil — Part 5
Hasilnya **identik dengan Expected Result** (`' the' 0.0965`, `' a' 0.0541`, …). Hal ini wajar karena forward pass tanpa sampling bersifat deterministik.

Yang menarik:
- Probabilitas tertinggi hanya **≈9,7%** dan kelima teratas berjumlah ≈**25%**. Distribusinya sangat **landai**: model tidak yakin.
- Token teratas adalah kata fungsi (`the`, `a`, `now`, `located`, `also`), **bukan "Jakarta"/"Nusantara"**. GPT-2 kecil tidak menyimpan fakta ini dengan kuat dan memprediksi kelanjutan *gramatikal* yang umum.
- Spasi ikut menjadi bagian token (`' the'`), ciri khas tokenizer BPE GPT-2.
- Inilah bahan mentah di balik `temperature`, `top_k`, dan `top_p` pada Part 1: semuanya hanya memodifikasi distribusi ini sebelum satu token dipilih.

---
---
# [Part 6] Parameter-Efficient Fine-Tuning with LoRA (`peft`)


Full fine-tuning of an LLM updates every weight parameter in the network, requiring massive GPU VRAM to store gradients and optimizer states. **LoRA (Low-Rank Adaptation)** solves this problem by:

1. **Freezing** all original pre-trained weights $W_0 \in \mathbb{R}^{d \times k}$ (`requires_grad = False`).
2. **Injecting** two small trainable low-rank matrices $A \in \mathbb{R}^{r \times k}$ and $B \in \mathbb{R}^{d \times r}$ (where rank $r \ll \min(d, k)$, e.g., $r = 8$) alongside target projection layers.
3. **Computing** the adapted forward pass as:

$$h = W_0 x + \frac{\alpha}{r} B A x$$



This typically reduces the number of trainable parameters to **less than 1% of the base model** while achieving comparable task performance!



When we pass both `inputs` and `labels` into a Hugging Face causal language model (`outputs = lora_model(**batch, labels=batch["input_ids"])`), the model's `forward()` method automatically shifts the labels by one token to the right so each position predicts the next token, and computes the standard **Cross-Entropy Loss** (`outputs.loss`).


In [19]:
# Upgrade torchao to a compatible version (> 0.16.0) to satisfy peft dependencies
!pip install -q "torchao>=0.16.0"


In this exercise, we will fine-tune a fresh copy of `gpt2` on a small custom domain dataset of **University AI Lab FAQs** so the model learns institutional knowledge not present in its pre-training data:


In [20]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

# Custom domain FAQ dataset
faq_corpus = [
    "Q: What is the GPU cluster quota for CS-402 students? A: Each student receives 20 GPU hours per week on the Ada-Lovelace cluster.",
    "Q: How do I request access to the university AI lab servers? A: Submit Form AI-101 signed by your thesis advisor to the IT helpdesk.",
    "Q: Where are the weekly PyTorch office hours held? A: Every Thursday from 3 PM to 5 PM in Engineering Building Room 304.",
    "Q: What is the late submission penalty for deep learning labs? A: Submissions lose 10 percent per day up to a maximum of three days."
]

# Load a fresh base GPT-2 model for LoRA adaptation
# Note: model_gpt2 and tokenizer_gpt2 are used here because they were loaded in Section 1
base_gpt2_for_lora = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

Create `lora_config` using `LoraConfig(...)`

Define a `LoraConfig` with `task_type=TaskType.CAUSAL_LM`, `r=8`, `lora_alpha=16`, `lora_dropout=0.05`, and `target_modules=["c_attn"]` (GPT-2's fused attention projection layer).

In [21]:
lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 8,
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    target_modules = ["c_attn"],
    fan_in_fan_out = True,
)

### Penjelasan Kode — `LoraConfig`
- `r=8`: rank matriks kecil A (8×k) dan B (d×8).
- `lora_alpha=16`: faktor skala; kontribusi adaptor = `α/r = 2` kali `B·A·x`.
- `lora_dropout=0.05`: dropout ringan pada jalur LoRA untuk mengurangi overfitting.
- `target_modules=["c_attn"]`: lapisan proyeksi Q/K/V gabungan milik GPT-2 yang diberi adaptor.
- `fan_in_fan_out=True`: GPT-2 memakai layer `Conv1D` yang bobotnya tersimpan tertukar (transposed) dibanding `nn.Linear`; opsi ini memberi tahu PEFT agar menanganinya.

Wrap `base_gpt2_for_lora` with `get_peft_model(...)` and print trainable parameters

Wrap the base model using `get_peft_model(base_gpt2, lora_config)` and call `.print_trainable_parameters()`.

In [22]:
lora_model = get_peft_model(base_gpt2_for_lora, lora_config)

lora_model.print_trainable_parameters()

W1125 01:09:52.122000 2223 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


### Analisis Hasil — Jumlah Parameter
`trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364`

Bisa diverifikasi manual: tiap dari 12 blok punya satu `c_attn` (768 → 2304). Adaptor-nya berukuran `8×768 + 2304×8 = 24.576` parameter, sehingga `12 × 24.576 = 294.912` ✔. Artinya kita hanya melatih **≈0,24%** parameter; sisanya (≈124,4 juta) dibekukan. Ini yang membuat LoRA hemat memori GPU (tidak perlu menyimpan gradien dan state optimizer untuk seluruh model) dan adaptornya kecil (bisa disimpan/dibagikan terpisah dari model dasar). Peringatan `KernelPreference ... register_constant()` berasal dari library pendukung (torchao/torch) dan bisa diabaikan.

---
#### <font color='red'>**EXERCISE:** </font>

    Complete the standard PyTorch training loop


In [23]:
# Initialize `torch.optim.AdamW()` on `lora_model.parameters()` with `lr=5e-4`
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=5e-4)

# Training loop
if lora_model is not None and optimizer is not None:

    lora_model.train()

    for epoch in range(25):
        total_loss = 0.0

        for text in faq_corpus:

            # Pass text and return_tensor="pt" through tokenizer_gpt2(), then call .to(device)
            inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)

            # Pass **inputs through lora_model() with labels=inputs["inputs_ids"]
            outputs = lora_model(**inputs, labels=inputs["input_ids"])

            # get training loss
            loss = outputs.loss

            # call .zero_grad() function from optimizer
            optimizer.zero_grad()

            # call .backward() function from loss
            loss.backward()

            # call .step() function from optimizer
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d}/25 | Average Loss: {total_loss / len(faq_corpus):.4f}")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 05/25 | Average Loss: 4.2150
Epoch 10/25 | Average Loss: 2.9925
Epoch 15/25 | Average Loss: 2.0272
Epoch 20/25 | Average Loss: 1.1027
Epoch 25/25 | Average Loss: 0.8513


### Penjelasan Kode — Loop Training
Ini adalah loop PyTorch standar, tiap sampel kalimat sebagai satu langkah (batch size 1):
1. `AdamW(lora_model.parameters(), lr=5e-4)`: karena bobot dasar dibekukan, hanya parameter LoRA yang benar-benar memperoleh update.
2. `lora_model(**inputs, labels=inputs["input_ids"])`: pada *causal LM*, label = input itu sendiri. Model otomatis menggeser label satu posisi sehingga tiap token dilatih memprediksi token berikutnya, dan menghitung **cross-entropy loss**.
3. **`zero_grad()` → `backward()` → `step()`**: urutan wajib. Hapus gradien lama, hitung gradien baru, lalu perbarui bobot. Kalau `zero_grad()` terlupa, gradien menumpuk antar-langkah.
4. Loss dirata-ratakan tiap epoch dan dicetak setiap 5 epoch.

**Expected Result**:
<pre>
Training loss should start at around 4.2 and end at around 0.7 in 25 epochs

### 📊 Analisis Hasil — Training
| Epoch | 5 | 10 | 15 | 20 | 25 |
|---|---|---|---|---|---|
| Avg Loss | 4.22 | 2.99 | 2.03 | 1.10 | **0.85** |

Loss turun terus dari ≈4,2 ke ≈0,85, **sesuai pola ekspektasi soal** (≈4,2 → ≈0,7); angka akhirnya sedikit di atas contoh. Selisih kecil itu wajar karena inisialisasi matriks LoRA dan dropout bersifat acak, jadi angka loss berbeda tipis di tiap run (run sebelumnya berakhir di 0,78). Perhatikan bahwa penurunan terjadi pada **hanya 4 kalimat yang diulang 25 kali**, jadi model pada dasarnya *menghafal* korpus FAQ. Untuk demo itu memang tujuannya (menanamkan fakta institusional yang tidak ada di pre-training), tetapi pada kasus nyata diperlukan data lebih banyak dan *validation set* untuk memantau overfitting. Loss yang rendah pada data latih **belum membuktikan** model menjawab dengan benar; itu dicek di sel berikutnya.

Test the fine-tuned LoRA model


In [24]:
lora_model.eval()

prompt = "Q: What is the GPU cluster quota for CS-402 students? A:"

print("--- Base model (GPT-2) Output ---")
print(generate_text(model_gpt2, tokenizer_gpt2, prompt, max_new_tokens=25, do_sample=False))

print("\n---  LoRA Fine-Tuned Output   ---")
print(generate_text(lora_model, tokenizer_gpt2, prompt, max_new_tokens=25, do_sample=False))


--- Base model (GPT-2) Output ---
Q: What is the GPU cluster quota for CS-402 students? A: The GPU cluster quota is the number of GPUs that are available for each student. The number of GPUs that are available for each

---  LoRA Fine-Tuned Output   ---
Q: What is the GPU cluster quota for CS-402 students? A: Each student receives 20 GPU hours per week on the Ada-Lovelace cluster. Each student receives 200 GPU hours per week


**Expected Result**:
<pre>
--- Base model (GPT-2) Output ---
Q: ...? A: The GPU cluster quota is the number of GPUs that are available for each student. The number of GPUs ...
---  LoRA Fine-Tuned Output   ---
Q: ...? A: Each student receives 20 GPU hours per week on cluster computing clusters. Each student receives up to ...

### 📊 Analisis Hasil — Uji Base vs LoRA
| | Jawaban setelah `A:` |
|---|---|
| **Base GPT-2** | "The GPU cluster quota is the number of GPUs that are available for each student. The number of GPUs that are available for each…" |
| **LoRA** | "Each student receives **20 GPU hours per week on the Ada-Lovelace cluster**. Each student receives 200 GPU hours per week" |

- **Fine-tuning berhasil menanamkan fakta.** Base GPT-2 tidak tahu kebijakan ini, jadi ia hanya mengulang-ulang definisi generik dan berputar. Model LoRA langsung mengeluarkan kalimat yang sama dengan data latih, termasuk detail yang mustahil ditebak ("20 GPU hours", "Ada-Lovelace cluster").
- **Tetapi ada gejala overfitting/halusinasi.** Setelah jawaban yang benar, model mengulang kalimat dan mengubah angkanya menjadi **"200" GPU hours**, yang salah. Penyebabnya: data hanya 4 kalimat, model kecil, greedy decoding tidak punya sinyal untuk berhenti, dan `max_new_tokens=25` memaksa model terus menulis. Untuk jawaban nyata kita perlu data lebih banyak/beragam, *end-of-sequence token* pada tiap contoh, atau pembatasan panjang output.
- Hasil ini sesuai dengan "Expected Result" soal, yang juga menunjukkan jawaban benar di awal lalu mengulang ("Each student receives up to …").
- Teks prompt ikut tercetak karena `generate_text` dipanggil dengan `with_input=True` (default).

---
---
# [Part 7] Retrieval-Augmented Generation (RAG)

LLMs suffer from two fundamental limitations:
1. **Static Knowledge Cutoff:** They do not know private company documents, recent events, or proprietary campus policies.
2. **Hallucinations:** When asked about unfamiliar facts, an LLM may fabricate plausible-sounding misinformation.

**Retrieval-Augmented Generation (RAG)** solves both issues without retraining the LLM! Instead, RAG connects the LLM to an external knowledge base in three stages:
1. **Indexing (Dense Embeddings):** <br>Pass every knowledge base document through a bi-encoder embedding model <br>(such as `sentence-transformers/all-MiniLM-L6-v2`) to convert text into dense semantic vectors $D \in \mathbb{R}^{N \times d}$. <br>(where $N$ is the number of documents and $d$ is the embedding dimension)
2. **Semantic Retrieval (Cosine Similarity in PyTorch):** <br>Encode the user's question into a query vector $q \in \mathbb{R}^{1 \times d}$ and compute the cosine similarity between $q$ and each document row $D_i$ using PyTorch matrix multiplication (`torch.matmul`):
$$\text{similarity}(q, D_i) = \frac{q \cdot D_i^T}{\|q\|_2 \|D_i\|_2}$$

3. **Grounded Generation:** <br>Inject the top-$k$ retrieved passages into the LLM's system/user prompt and instruct the model to answer **strictly using the provided context**.



In [25]:
from sentence_transformers import SentenceTransformer

# Load a fast, lightweight bi-encoder embedding model (~80 MB)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))

# Simulated University & Research Lab Knowledge Base
knowledge_base = [
    "Policy 101 (Lab Access): The Robotics & AI Lab on Floor 3 is open Monday to Saturday from 08:00 to 21:00 WIB. RFID student cards are required after 18:00.",
    "Policy 102 (GPU Cluster Booking): Students may reserve up to 2 NVIDIA A100 GPUs for 12 hours per week via the Slurm scheduler using the command 'sbatch --gres=gpu:2'.",
    "Policy 103 (Thesis Submission): Final undergraduate thesis PDFs must be uploaded to the Academic Portal at least 14 days prior to the defense seminar.",
    "Policy 104 (Travel Grants): Graduate students with an accepted paper at NeurIPS, ICML, or ICLR qualify for up to IDR 25,000,000 in conference travel funding.",
    "Policy 105 (Dataset Privacy): Medical or personally identifiable datasets must remain encrypted on local campus servers and never be uploaded to public cloud drives."
]

# Precompute document embeddings as a PyTorch tensor of shape (num_docs, embedding_dim)
doc_embeddings = embedder.encode(knowledge_base, convert_to_tensor=True, normalize_embeddings=True)


### Penjelasan Kode — Indexing (Embedding)
`SentenceTransformer("all-MiniLM-L6-v2")` adalah **bi-encoder**: mengubah satu kalimat menjadi vektor padat 384 dimensi sehingga kalimat yang *maknanya* mirip memiliki vektor yang berdekatan. Kelima kebijakan di `knowledge_base` di-encode sekali di depan menjadi tensor `doc_embeddings` berukuran `(5, 384)`. Opsi `normalize_embeddings=True` membuat setiap vektor berpanjang 1 (norma L2 = 1), sehingga **dot product = cosine similarity**. Ini yang dipakai pada langkah retrieval.

---
#### <font color='red'>**EXERCISE:** </font>

    complete the function below to perform semantic search over `knowledge_base`


In [26]:
def retrieve_top_k(query, k=2):
    # Encode `query` into a normalized PyTorch tensor of shape (1, embedding_dim)
    query_embedding = embedder.encode([query], convert_to_tensor=True, normalize_embeddings=True)

    # Compute cosine similarity scores between `query_embedding` and `doc_embeddings`
    # Since both tensors are L2-normalized, cosine similarity equals dot product
    # Pass query_embedding and doc_embeddings.T through torch.matmul(), then call .squeeze(0)
    scores = torch.matmul(query_embedding, doc_embeddings.T).squeeze(0)

    # Find the top-k highest scores
    # Pass scores through torch.topk() with k=k
    top_scores, top_indices = torch.topk(scores, k=k)

    results = []
    for score, idx in zip(top_scores, top_indices):
        results.append((score.item(), knowledge_base[idx.item()]))
    return results


### Penjelasan Kode — `retrieve_top_k`
Pertanyaan di-encode dengan model yang sama → `query_embedding (1, 384)`. Lalu `torch.matmul(query_embedding, doc_embeddings.T)` menghasilkan `(1, 5)` skor kemiripan dengan semua dokumen sekaligus (satu perkalian matriks, tanpa loop). `.squeeze(0)` membuangnya menjadi vektor `(5,)`, dan `torch.topk(scores, k)` memilih k dokumen paling relevan. Fungsi mengembalikan pasangan *(skor, teks dokumen)*.

---
#### <font color='red'>**EXERCISE:** </font>

    complete the function below to build an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline


In [27]:
def rag_answer(user_question, k=2):

    # Pass user_question and k=k through retrieve_top() function
    retrieved_docs = retrieve_top_k(user_question, k=k)

    context_block = "\n".join([f"- {doc}" for _, doc in retrieved_docs])

    messages = [
        {"role": "system",
         "content": (
             "You are a factual university assistant. Answer the user's question using ONLY "
             "the retrieved policy context below. If the answer is not in the context, say "
             "'I do not have that information in the knowledge base.'"
        )},
        {"role": "user",
         "content": f"Retrieved Context:\n{context_block}\n\nQuestion: {user_question}"
        }
    ]

    # Apply chat template
    # Pass messages, tokenize=False, and add_generation_prompt=True
    # through chat_tokenizer.apply_chat_template()
    formatted_prompt = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Call generate_text() with input chat_model, chat_tokenizer, and formatted_prompt
    # set max_new_tokens=100, do_sample=False, and with_input=False
    answer = generate_text(chat_model, chat_tokenizer, formatted_prompt,
                           max_new_tokens=100, do_sample=False, with_input=False)

    return answer, retrieved_docs

### Penjelasan Kode — `rag_answer` (pipeline RAG)
1. **Retrieve:** ambil k dokumen paling relevan (`retrieve_top_k`).
2. **Augment:** gabungkan dokumen menjadi `context_block` dan masukkan ke prompt `user` bersama pertanyaan.
3. **Generate:** system prompt memerintahkan model menjawab **HANYA** dari konteks dan, bila tidak ada jawabannya, mengucapkan kalimat penolakan baku. Greedy decoding (`do_sample=False`) dipakai agar jawaban faktual dan konsisten.

Dengan cara ini pengetahuan privat (kebijakan kampus) dipakai **tanpa melatih ulang** model, dan sumber jawabannya bisa ditampilkan (transparan). Komentar soal menulis `retrieve_top()`, padahal nama fungsinya `retrieve_top_k()`.

Let's test your implementation with  Question answerable from the knowledge base

In [28]:
q1 = "How many GPUs can a student reserve per week and what Slurm command is used?"
ans1, docs1 = rag_answer(q1, k=2)

print(f"Question 1: {q1}")
print("Retrieved Sources:")
for score, doc in docs1:
    print(f"  [Score: {score:.3f}] {doc}")
print(f"RAG Answer: {ans1}\n")

Question 1: How many GPUs can a student reserve per week and what Slurm command is used?
Retrieved Sources:
  [Score: 0.773] Policy 102 (GPU Cluster Booking): Students may reserve up to 2 NVIDIA A100 GPUs for 12 hours per week via the Slurm scheduler using the command 'sbatch --gres=gpu:2'.
  [Score: 0.212] Policy 105 (Dataset Privacy): Medical or personally identifiable datasets must remain encrypted on local campus servers and never be uploaded to public cloud drives.
RAG Answer: A student can reserve up to 2 NVIDIA A100 GPUs for 12 hours per week via the Slurm scheduler using the command 'sbatch --gres=gpu:2'.



**Expected Result**:
<pre>
Retrieved Sources:
  [Score: 0.773] Policy 102 (GPU Cluster Booking): Students may reserve up to ...
  [Score: 0.212] Policy 105 (Dataset Privacy): Medical or personally identifiable datasets ...
RAG Answer: A student can reserve up to 2 NVIDIA A100 GPUs for 12 hours per week via the Slurm scheduler using the command 'sbatch --gres=gpu:2'.

### 📊 Analisis Hasil — RAG Pertanyaan #1
- **Retrieval tepat:** Policy 102 (GPU Cluster Booking) berada di peringkat 1 dengan skor **0,773**, sedangkan dokumen peringkat 2 (Policy 105, privasi dataset) hanya **0,212**. Selisih sebesar ini menunjukkan *retriever* yakin dan angkanya **identik dengan Expected Result**.
- **Jawaban benar dan grounded:** model menyebut **2 GPU NVIDIA A100, 12 jam per minggu, perintah `sbatch --gres=gpu:2`**, persis seperti isi dokumen. Dokumen kedua yang tidak relevan **diabaikan** oleh model. Jadi `k=2` yang memaksa mengambil dokumen kurang relevan tidak merusak jawaban.
- Catatan: jawabannya hampir menyalin kalimat dokumen. Itu wajar untuk RAG berinstruksi "gunakan HANYA konteks" dan greedy decoding.

Let's test it with Out-of-scope question to verify hallucination resistance

In [29]:
q2 = "What is the cafeteria menu for Friday lunch?"
ans2, _ = rag_answer(q2, k=2)

print(f"Question 2: {q2}")
print(f"RAG Answer: {ans2}")


Question 2: What is the cafeteria menu for Friday lunch?
RAG Answer: I do not have that information in the knowledge base.


**Expected Result**:
<pre>
RAG Answer: I do not have that information in the knowledge base.

### Analisis Hasil — RAG Pertanyaan #2 (di luar cakupan)
Jawaban: *"I do not have that information in the knowledge base."* → **sesuai Expected Result**. Menu kantin tidak ada di knowledge base, dan model patuh pada instruksi untuk tidak mengarang. Inilah nilai utama RAG: **mengurangi halusinasi** dengan menambatkan jawaban pada dokumen. Perlu diingat bahwa `retrieve_top_k` *selalu* mengembalikan k dokumen walaupun tidak relevan; yang mencegah halusinasi di sini adalah **instruksi pada system prompt**, bukan retrieval-nya. Untuk sistem produksi sebaiknya ditambah ambang skor minimum (mis. abaikan dokumen dengan skor < 0,3).

---
# 🔍 Review Hasil Run

Seluruh 29 sel kode berhasil dijalankan berurutan (execution count 1–29) dan **tidak ada error**.

| Part | Status | Catatan |
|---|---|---|
| 0 Environment | ✅ | GPU CUDA + float16 terdeteksi |
| 1 GPT-2 | ✅ | Greedy berulang, sampling bervariasi dan berubah tiap run (sesuai teori) |
| 2 Chatbot | ✅ / ⚠️ | Memori multi-turn terbukti walau jawaban giliran 2 ragu-ragu; ada halusinasi "ITB"; chat interaktif baru diuji dengan `exit` |
| 3 Sentimen | ✅ | 3/3 benar |
| 4 JSON | ✅ / ⚠️ | Kasus #1 sempurna; kasus #2 ada halusinasi `credits=3` dan field `instructor` berisi kalimat |
| 5 Probabilitas | ✅ | Angka identik dengan Expected Result |
| 6 LoRA | ✅ | 294.912 parameter (0,24%); loss 4,22 → 0,85; LoRA menjawab fakta yang benar, base GPT-2 tidak, tetapi ada pengulangan dengan angka salah ("200") |
| 7 RAG | ✅ | Skor retrieval 0,773 / 0,212 identik dengan Expected Result; jawaban benar; pertanyaan di luar cakupan ditolak dengan tepat |

Peringatan yang muncul (HF token, `loss_type=None`, `KernelPreference`) hanya informasi, bukan kegagalan.

**Sisa yang masih kurang (opsional):**
1. Perintah `/reset` belum diuji; chat interaktif hanya diuji dengan `exit`.
2. Masalah halusinasi di Part 4 (#2) belum diperbaiki; cukup tambahkan instruksi `null` untuk field yang tidak ada di prompt dan jalankan ulang.
3. Evaluasi masih berskala kecil (3 ulasan, 2 pertanyaan RAG), sehingga belum bersifat kuantitatif.

---
# 🧾 Kesimpulan

1. **LLM adalah mesin prediksi token berikutnya.** Semua yang kita lakukan, dari chat sampai JSON, adalah cara berbeda memanfaatkan satu mekanisme yang sama: menghitung distribusi probabilitas atas kosakata (Part 5) lalu memilih token dengan strategi tertentu (Part 1).
2. **Decoding menentukan karakter output.** Greedy bersifat deterministik tetapi rawan berulang; sampling lebih variatif tetapi bisa melantur dan hasilnya berubah tiap run (terlihat jelas di Part 1 dan Part 2). Pilih sesuai tugas: greedy untuk klasifikasi/ekstraksi/QA, sampling untuk teks kreatif.
3. **Model instruksi + chat template mengubah LLM menjadi asisten.** LLM *stateless*, sehingga "ingatan" chatbot adalah riwayat pesan yang dikirim ulang (dengan konsekuensi konteks yang makin panjang). Model kecil tetap bisa mengarang detail (mis. kepanjangan "ITB") dan menjawab ragu-ragu.
4. **Prompt engineering sudah bisa menggantikan banyak training.** Few-shot (Part 3) mencapai 3/3 benar tanpa mengubah bobot, dan skema JSON (Part 4) menghasilkan data terstruktur. Namun Part 4 juga menunjukkan risikonya: **halusinasi** saat informasi tidak ada, sehingga perlu instruksi *fallback* `null`, validasi, dan penanganan error.
5. **LoRA membuat fine-tuning terjangkau.** Hanya ≈0,24% parameter yang dilatih, namun loss turun drastis (≈4,2 → ≈0,85) dan model berhasil mengeluarkan fakta domain yang tidak diketahui base GPT-2 ("20 GPU hours per week"). Kelemahannya terlihat juga: pengulangan dan angka yang salah ("200") akibat data sangat sedikit, dan pengetahuan yang ditanam ke bobot sulit diperbarui.
6. **RAG adalah alternatif yang lebih murah dan mudah diperbarui.** Pengetahuan diletakkan di luar model (embedding + cosine similarity), disisipkan ke prompt saat dibutuhkan. Hasil run menunjukkan retrieval akurat (skor 0,773 vs 0,212), jawaban sesuai dokumen, dan pertanyaan di luar cakupan ditolak, sehingga halusinasi berkurang dan sumber bisa dilacak.
7. **LoRA vs RAG:** gunakan *RAG* untuk fakta yang sering berubah atau perlu sumber; gunakan *LoRA* untuk mengubah **gaya/perilaku/format** model. Keduanya dapat digabungkan. Pada eksperimen ini RAG memberi jawaban yang lebih bersih daripada LoRA untuk menjawab fakta.

**Keterbatasan:** semua model di sini kecil (GPT-2 124M, Qwen 1,5B), jumlah uji sangat sedikit, dan sebagian hasil bergantung pada sampling acak sehingga berubah antar-run. Kesimpulan di atas bersifat ilustratif, bukan evaluasi kuantitatif.

---
---

# Congratulation, You've Completed Exercise 8

<p>Copyright &copy;  <a href=https://www.linkedin.com/in/andityaarifianto/>2026 - ADF</a> </p>

![footer](https://i.imgur.com/UU76cTP.png)